Read input data

In [1]:
from pandas_plink import read_plink
import pandas as pd

## Plink files
(bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)

## Read base data
base_data = pd.read_csv("../tests/data/Height.QC.gz", sep="\t", compression="gzip")

## Read covariate data
covariate_data = pd.read_csv("../tests/data/EUR.covariate", sep=" ")

/tmp/ipykernel_1057/1365574468.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  (bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)
/tmp/ipykernel_1057/1365574468.py:5: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  (bim, fam, G) = read_plink("../tests/data/EUR.QC", verbose=False)


In [2]:
command = {
    "binary-target": "F",
    "base-maf": "MAF:0.01",
    "base-info": "INFO:0.8",
    "stat": "OR",
    "or": True,
    "out": "EUR"
}

In [5]:
bim

,chrom,snp,cm,pos,a0,a1,i
0,1,rs3131962,0.490722,756604,A,G,0
1,1,rs4040617,0.500708,779322,G,A,1
2,1,rs79373928,0.587220,801536,G,T,2
3,1,rs11240779,0.620827,808631,G,A,3
4,1,rs57181708,0.620827,809876,G,A,4
...,...,...,...,...,...,...,...
489800,22,rs73174435,75.082500,51174939,T,C,489800
489801,22,rs3810648,75.083200,51175626,G,A,489801
489802,22,rs5771002,75.089100,51183255,A,G,489802
489803,22,rs3865764,75.091100,51185848,G,A,489803


In [23]:
base_data

,CHR,BP,SNP,A1,A2,N,SE,P,OR,INFO,MAF
0,1,756604,rs3131962,A,G,388028,0.003017,0.483171,0.997887,0.890558,0.369390
1,1,768448,rs12562034,A,G,388028,0.003295,0.834808,1.000687,0.895894,0.336846
2,1,779322,rs4040617,G,A,388028,0.003033,0.428970,0.997604,0.897508,0.377368
3,1,801536,rs79373928,G,T,388028,0.008413,0.808999,1.002036,0.908963,0.483212
4,1,808631,rs11240779,G,A,388028,0.002428,0.590265,1.001308,0.893213,0.450410
...,...,...,...,...,...,...,...,...,...,...,...
499612,22,51174939,rs73174435,T,C,388028,0.004538,0.007335,1.012243,0.887884,0.487075
499613,22,51175626,rs3810648,G,A,388028,0.004251,0.000078,1.016933,0.890887,0.290302
499614,22,51183255,rs5771002,A,G,388028,0.002156,0.000799,1.007256,0.890567,0.237780
499615,22,51185848,rs3865764,G,A,388028,0.004631,0.554611,0.997267,0.905337,0.403402


In [ ]:
BITCT = 64
BITCT2 = BITCT / 2
VEC_BYTES = 16
VEC_BITS = VEC_BYTES * 8
VEC_WORDS = VEC_BITS / BITCT

BITCT_TO_VECCT = lambda val: (((val) + (VEC_BITS - 1)) / VEC_BITS)
BITCT_TO_ALIGNED_WORDCT = lambda val: VEC_WORDS * BITCT_TO_VECCT(val)
BITCT_TO_WORDCT = lambda val: (((val) + (BITCT - 1)) / BITCT)
QUATERCT_TO_VECCT = lambda val: (((val) + ((VEC_BITS / 2) - 1)) / (VEC_BITS / 2))

QUATERCT_TO_WORDCT = lambda val: (((val) + (BITCT2 - 1)) / BITCT2)
QUATERCT_TO_ALIGNED_WORDCT = lambda val: (VEC_WORDS * QUATERCT_TO_VECCT(val))


class Genotype:

    def __init__(self, plink_files, base_file, cov_file):
        self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)
        self.base_data = pd.read_csv(base_file, sep="\t", compression="gzip")
        self.covariate_data = pd.read_csv(cov_file, sep=" ")

    @property
    def m_sort_by_p_index(self):
        """
        Sort by: chr ↑ → p_value ↑ → loc ↑ → rs ↑
        """
        # inc/genotype.hpp:139
        idx = self.bim.merge(
                    self.base_data[["SNP", "P", "CHR", "BP"]], 
                    left_on="snp", 
                    right_on="SNP", 
                    how="left")\
                .sort_values(by=["CHR", "P", "BP", "SNP"])\
                .index.to_list()                # src/snp.cpp:27
        
        return idx

    def get_chrom_boundary(self):
        # src/genotype.cpp:1171
        dta = self.bim.loc[:,"chrom"].value_counts()\
                        .reset_index()\
                        .astype({"chrom": int})\
                        .sort_values(by="chrom")
        dta["second"] = dta["count"].cumsum()
        dta["first"] = dta["second"].shift(1, fill_value=0)
        dta_list = dta.loc[:,["first", "second"]].values.tolist()

        return dta_list

    @property
    def m_max_window_size(self):
        # src/genotype.cpp:99 & src/genotype.cpp:111
        # src/genotype.cpp:83
        pass

    def clumping(self):
        # src/genotype.cpp:1194
        # src/genotype.cpp:1274
        min_r2 = 0.1
        m_founder_ct = len(self.fam)
        m_unfiltered_sample_ct = len(self.fam)
        founder_ctv3 = BITCT_TO_ALIGNED_WORDCT(m_founder_ct)
        founder_ctl2 = QUATERCT_TO_WORDCT(m_founder_ct)

        founder_ctsplit = 3 * founder_ctv3
        founder_ctv2 = QUATERCT_TO_ALIGNED_WORDCT(m_founder_ct)
        unfiltered_sample_ctl = BITCT_TO_WORDCT(m_unfiltered_sample_ct)

        unfiltered_sample_ctv2 = 2 * unfiltered_sample_ctl
        index_data = [0] * 3 * founder_ctsplit + founder_ctv3
        index_tots = [0] * 6

        founder_include2 = [0] * founder_ctv2

        snp_range = self.get_chrom_boundary()            # src/genotype.cpp:1200

        num_snp_in_chr = sum([i[1] - i[0] for i in snp_range])
        max_snp_in_chr = 0

        # src/genotype.cpp:1300
        # m_max_window_size


        pass

gt = Genotype(
        "../tests/data/EUR.QC",
        "../tests/data/Height.QC.gz",
        "../tests/data/EUR.covariate"
    )

gt.get_chrom_boundary()

/tmp/ipykernel_1057/1290974401.py:19: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)
/tmp/ipykernel_1057/1290974401.py:19: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.bim, self.fam, self.G = read_plink(plink_files, verbose=False)


[[0, 39084],
 [39084, 78654],
 [78654, 111976],
 [111976, 143284],
 [143284, 172724],
 [172724, 206708],
 [206708, 233709],
 [233709, 259004],
 [259004, 280344],
 [280344, 305092],
 [305092, 329364],
 [329364, 352273],
 [352273, 369393],
 [369393, 385379],
 [385379, 400833],
 [400833, 417745],
 [417745, 433499],
 [433499, 448451],
 [448451, 461270],
 [461270, 474414],
 [474414, 481911],
 [481911, 489805]]